In [1]:
# 1. Install required packages
!pip install google-genai gradio opencv-python-headless numpy pandas scikit-learn

import os
import cv2
import numpy as np
import pandas as pd
import datetime
import random
import time
import joblib
from sklearn.ensemble import RandomForestClassifier

# 2. IMAGE PROCESSING LAYER (Reading both 'yes' and 'no' folders)
X_images = []
y_labels = []

# Paths to your local folders in Colab
no_folder = '/content/no'
yes_folder = '/content/yes'

# Process Normal Scans (Label = 0)
if os.path.exists(no_folder):
    print(f"📁 Processing normal images from '{no_folder}'...")
    for file_name in os.listdir(no_folder):
        img = cv2.imread(os.path.join(no_folder, file_name), cv2.IMREAD_GRAYSCALE)
        if img is None: continue
        img_resized = cv2.resize(img, (64, 64))
        X_images.append(img_resized.flatten())
        y_labels.append(0)

# Process Tumor Scans (Label = 1)
if os.path.exists(yes_folder):
    print(f"📁 Processing tumor images from '{yes_folder}'...")
    for file_name in os.listdir(yes_folder):
        img = cv2.imread(os.path.join(yes_folder, file_name), cv2.IMREAD_GRAYSCALE)
        if img is None: continue
        img_resized = cv2.resize(img, (64, 64))
        X_images.append(img_resized.flatten())
        y_labels.append(1) # Label 1 = Tumor Detected

X = np.array(X_images)
y = np.array(y_labels)

# 3. Train the ML Brain on both classes
if len(X) > 0:
    brain_model = RandomForestClassifier(random_state=42)
    brain_model.fit(X, y)
    joblib.dump(brain_model, 'brain_tumor_model.pkl')
    print(f"🧠 System Message: Machine Learning Brain trained successfully on {len(X)} total images (Tumor & Normal)!")
else:
    print("⚠️ Error: No images were found in /content/no or /content/yes. Check your folder sidebar!")

📁 Processing normal images from '/content/no'...
📁 Processing tumor images from '/content/yes'...
🧠 System Message: Machine Learning Brain trained successfully on 75 total images (Tumor & Normal)!


In [ ]:
from google import genai
from google.genai import types

# 1. INITIALIZE AGENT BRAIN
# Replace 'YOUR_API_KEY' with your real key from Google AI Studio
client = genai.Client(api_key='enter_your_gemini_api_here')

# 2. DEFINING THE MACHINE LEARNING TOOL FOR THE AGENT
def classify_mri_scan(image_path: str) -> str:
    """
    Analyzes a raw brain MRI scan image file and returns a machine learning
    prediction indicating if a tumor is detected or if the scan is normal.
    """
    if not image_path:
        return "Error: No image provided."

    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return "Error: Invalid image file path."

    img_resized = cv2.resize(img, (64, 64))
    flattened_img = img_resized.flatten()

    # Predict using the model trained in Cell 1
    prediction = brain_model.predict([flattened_img])[0]

    if prediction == 1:
        return "ML Model Result: ⚠️ TUMOR DETECTED"
    else:
        return "ML Model Result: ✅ NORMAL (No tumor patterns detected)"

print("🤖 System Message: Agent Tools and Reasoning Brain initialized.")

🤖 System Message: Agent Tools and Reasoning Brain initialized.


In [8]:
import gradio as gr

tumor_agent_history = []

# 1. CORE AGENT LOGIC FUNCTION
def ai_agent_clinic_handler(patient_id, scan_type, mri_image_path):
    global tumor_agent_history

    if mri_image_path is None:
        return "🤖 Agent: Please upload an MRI scan image.", pd.DataFrame(tumor_agent_history)

    system_instruction = (
        "You are an autonomous Medical AI Radiologist Assistant. "
        "You have access to a tool called `classify_mri_scan`. When a doctor provides a patient scan, "
        "you must use this tool to see what the machine learning model thinks, then synthesize a "
        "professional, detailed medical report with clinical protocols based on its findings."
    )

    doctor_prompt = (
        f"Please analyze this new patient case.\n"
        f"Patient ID: {patient_id}\n"
        f"Scan Modality: {scan_type}\n"
        f"The image file is located at: {mri_image_path}"
    )


   # Agent evaluates the instructions, calls tool, and responds
    response = client.models.generate_content(
        model='gemini-2.5-flash',  # <-- Changed this from 2.5 to 2.0
        contents=doctor_prompt,
        config=types.GenerateContentConfig(
            system_instruction=system_instruction,
            tools=[classify_mri_scan],
            temperature=0.2
        )
    )


    report_text = response.text
    # A more precise way to check what conclusion the agent reached
    if "NORMAL" in report_text.upper() and "⚠️ TUMOR DETECTED" not in report_text.upper():
        agent_decision = "Normal"
    else:
        agent_decision = "Tumor Flagged"
    # Update ledger log
    timestamp_str = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    tumor_agent_history.append({
        "Timestamp": timestamp_str,
        "Patient ID": patient_id,
        "Scan Modality": scan_type,
        "Agent Conclusion": agent_decision,
        "Agent Narrative": report_text[:120] + "..."
    })

    return report_text, pd.DataFrame(tumor_agent_history)

# 2. GRADIO INTERFACE LAYOUT
with gr.Blocks(title="Brain Tumor AI Agent Workspace") as demo:
    gr.Markdown("# 🧠 Autonomous Brain Tumor Classification & Analysis Agent")
    gr.Markdown("### **Specialization:** Computer Engineering Portfolio Submission")

    with gr.Row():
        with gr.Column():
            gr.Markdown("### 📥 Patient & Scan Metadata Inputs")
            p_id = gr.Textbox(label="Patient Case Identifier", value="CASE-9842")
            s_type = gr.Dropdown(choices=["MRI T1-Weighted", "MRI T2-Weighted", "MRI T2-FLAIR"], label="Scan Modality Sequence", value="MRI T2-FLAIR")
            input_img = gr.Image(label="Upload Brain MRI Image Matrix", type="filepath")
            btn_analyze = gr.Button("Execute Vision Core Diagnostics", variant="primary")

        with gr.Column():
            gr.Markdown("### 📝 System Output Terminal")
            agent_report = gr.Markdown("*Awaiting image upload execution matrix...*")

    gr.Markdown("---")
    gr.Markdown("### 🗄️ Unified Clinical Transaction History Ledger")
    ledger_table = gr.DataFrame(label="System State Database Log", value=pd.DataFrame(tumor_agent_history))

    # Binding action trigger to agent handler
    btn_analyze.click(
        fn=ai_agent_clinic_handler,
        inputs=[p_id, s_type, input_img],
        outputs=[agent_report, ledger_table]
    )

# Launch app with a shareable public URL
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://a1dd6efd7002ef30cc.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://a1dd6efd7002ef30cc.gradio.live
